# 01 - Data Loading & Exploratory Data Analysis (EDA)
## PhishScamSense: Real-Time Multimodal Phishing Defense

This notebook covers:
1. Loading the **CIC-Bell-DNS2021** dataset (benign, phishing, malware, spam)
2. Dataset statistics and 4-class distribution
3. URL length and structural feature analysis
4. Per-class pattern comparison (TLD, entropy, special characters)

In [ ]:
import sys
import os

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, PROJECT_ROOT)

import logging
from pathlib import Path
from collections import Counter
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from ml.src.data.data_loader import load_cic_bell_dns2021
from ml.src.features.url_features import extract_url_features

sns.set_theme(style="whitegrid")
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Consistent color palette for 4 classes
CLASS_NAMES  = ["benign", "phishing", "malware", "spam"]
CLASS_COLORS = ["#2ecc71", "#e74c3c", "#e67e22", "#9b59b6"]
CLASS_PALETTE = dict(zip(CLASS_NAMES, CLASS_COLORS))

DATA_DIR = Path(PROJECT_ROOT) / "data" / "raw"
print(f"Project root : {PROJECT_ROOT}")
print(f"Data dir     : {DATA_DIR}")

## 1.1 Load CIC-Bell-DNS2021 Dataset

Four raw CSV files in `data/raw/`:

| File | Format | Label |
|---|---|---|
| `benign_domains.csv` | one URL per line | 0 |
| `phishing_domains.csv` | PhishTank CSV — URL at column 1 | 1 |
| `malware_domains.csv` | one URL per line | 2 |
| `spam_domains.csv` | one URL per line | 3 |

`max_benign=100_000` subsamples from ~988k benign entries to keep EDA manageable.

In [ ]:
# Load full dataset (benign capped at 100k; all other classes fully loaded)
urls, labels = load_cic_bell_dns2021(DATA_DIR, max_benign=100_000, seed=42)

df_full = pd.DataFrame({"url": urls, "label": labels})
df_full["label_name"] = df_full["label"].map(dict(enumerate(CLASS_NAMES)))

print(f"Total samples : {len(df_full):,}")
print(f"\nClass distribution (full dataset):")
counts = df_full["label_name"].value_counts().reindex(CLASS_NAMES)
for name, cnt in counts.items():
    pct = cnt / len(df_full) * 100
    print(f"  {name:12s}: {cnt:>8,}  ({pct:5.1f}%)")

## 1.2 EDA Sample

For interactive EDA we use a stratified sample (up to 5,000 per class) so cells run quickly.  
All counts and percentages in section 1.1 above reflect the full loaded dataset.

In [ ]:
SAMPLE_PER_CLASS = 5_000

df = (
    df_full
    .groupby("label", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), SAMPLE_PER_CLASS), random_state=42))
    .reset_index(drop=True)
)

print(f"EDA sample size: {len(df):,}")
print(df["label_name"].value_counts().reindex(CLASS_NAMES).to_string())
df.head()

## 1.3 URL Feature Extraction

In [ ]:
# Extract lexical features for each URL in the EDA sample
print("Extracting URL features…")
feature_rows = []
for url in df["url"]:
    try:
        feature_rows.append(extract_url_features(url))
    except Exception:
        feature_rows.append({})

df_feat = pd.DataFrame(feature_rows).fillna(0)
df_feat["label"]      = df["label"].values
df_feat["label_name"] = df["label_name"].values
df_feat["url"]        = df["url"].values

# Also parse scheme/hostname for TLD analysis
df_feat["scheme"]   = df["url"].apply(lambda u: urlparse(u).scheme)
df_feat["hostname"] = df["url"].apply(lambda u: urlparse(u).hostname or "")
df_feat["tld"]      = df_feat["hostname"].apply(lambda h: h.split(".")[-1] if "." in h else "")

print(f"Features extracted: {len(df_feat.columns) - 4} URL features across {len(df_feat):,} samples")
df_feat[["url", "label_name", "url_length", "num_dots", "url_entropy", "has_https"]].head(8)

In [ ]:
## 1.4 Class Distribution

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Bar chart (full dataset counts) ---
full_counts = df_full["label_name"].value_counts().reindex(CLASS_NAMES)
bars = axes[0].bar(CLASS_NAMES, full_counts.values, color=CLASS_COLORS, edgecolor="white", linewidth=0.8)
axes[0].set_title("Class Distribution — Full Dataset", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Sample Count")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for bar, val in zip(bars, full_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 400,
                 f"{val:,}", ha="center", va="bottom", fontsize=9)

# --- Pie chart ---
axes[1].pie(
    full_counts.values,
    labels=CLASS_NAMES,
    colors=CLASS_COLORS,
    autopct="%1.1f%%",
    startangle=140,
    wedgeprops={"edgecolor": "white", "linewidth": 1.2},
)
axes[1].set_title("Class Proportion — Full Dataset", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

print("\nNote: benign is capped at 100,000 (raw file has ~988k entries).")

In [ ]:
## 1.5 URL Length & Structural Features by Class

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# --- URL length histogram ---
for name, color in CLASS_PALETTE.items():
    subset = df_feat[df_feat["label_name"] == name]["url_length"]
    axes[0, 0].hist(subset.clip(upper=300), bins=40, alpha=0.55, label=name, color=color)
axes[0, 0].set_title("URL Length Distribution")
axes[0, 0].set_xlabel("URL Length (clipped at 300)")
axes[0, 0].legend()

# --- URL entropy box plot ---
sns.boxplot(
    data=df_feat, x="label_name", y="url_entropy",
    order=CLASS_NAMES, palette=CLASS_PALETTE, ax=axes[0, 1],
    linewidth=0.8,
)
axes[0, 1].set_title("URL Shannon Entropy")
axes[0, 1].set_xlabel("")

# --- Number of dots box plot ---
sns.boxplot(
    data=df_feat, x="label_name", y="num_dots",
    order=CLASS_NAMES, palette=CLASS_PALETTE, ax=axes[1, 0],
    linewidth=0.8,
)
axes[1, 0].set_title("Number of Dots in URL")
axes[1, 0].set_xlabel("")

# --- Number of hyphens box plot ---
sns.boxplot(
    data=df_feat, x="label_name", y="num_hyphens",
    order=CLASS_NAMES, palette=CLASS_PALETTE, ax=axes[1, 1],
    linewidth=0.8,
)
axes[1, 1].set_title("Number of Hyphens in URL")
axes[1, 1].set_xlabel("")

plt.suptitle("Structural URL Features by Class (EDA Sample)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
## 1.6 HTTPS Usage & TLD Analysis by Class

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- HTTPS usage stacked bar ---
https_counts = (
    df_feat.groupby(["label_name", "scheme"])
    .size()
    .unstack(fill_value=0)
    .reindex(CLASS_NAMES)
)
https_counts.plot(
    kind="bar", ax=axes[0],
    color=["#c0392b", "#27ae60"],
    edgecolor="white", linewidth=0.6,
)
axes[0].set_title("HTTP vs HTTPS Usage by Class")
axes[0].set_ylabel("Count")
axes[0].set_xlabel("")
axes[0].set_xticklabels(CLASS_NAMES, rotation=0)
axes[0].legend(title="Scheme")

# --- Top 8 TLDs per class (heatmap) ---
top_tlds = df_feat["tld"].value_counts().head(12).index.tolist()
tld_matrix = (
    df_feat[df_feat["tld"].isin(top_tlds)]
    .groupby(["label_name", "tld"])
    .size()
    .unstack(fill_value=0)
    .reindex(CLASS_NAMES)
    .fillna(0)
)
# Normalise per class so scale differences don't hide patterns
tld_norm = tld_matrix.div(tld_matrix.sum(axis=1), axis=0)
sns.heatmap(
    tld_norm, ax=axes[1], cmap="YlOrRd",
    linewidths=0.4, annot=True, fmt=".2f",
    cbar_kws={"label": "Proportion within class"},
)
axes[1].set_title("Top TLD Distribution by Class (row-normalised)")
axes[1].set_xlabel("TLD")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
## 1.7 Summary Statistics & Feature Correlation

# Per-class mean for all numerical features
numeric_cols = [
    "url_length", "hostname_length", "path_length",
    "num_dots", "num_hyphens", "num_digits",
    "url_entropy", "hostname_entropy",
    "has_ip_address", "has_https", "has_at_symbol",
    "subdomain_count", "tld_length", "vowel_ratio",
]
summary = df_feat.groupby("label_name")[numeric_cols].mean().T.reindex(columns=CLASS_NAMES)
print("=== Per-class feature means (EDA sample) ===\n")
print(summary.round(3).to_string())

# --- Correlation heatmap (all classes combined) ---
fig, ax = plt.subplots(figsize=(12, 9))
corr = df_feat[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, ax=ax, cmap="coolwarm",
    center=0, linewidths=0.3, annot=True, fmt=".2f", annot_kws={"size": 7},
    vmin=-1, vmax=1,
)
ax.set_title("Feature Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Save EDA sample with features to data/processed/ for downstream notebooks
PROCESSED_DIR = Path(PROJECT_ROOT) / "data" / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)
out_path = PROCESSED_DIR / "eda_sample_with_features.csv"
df_feat.to_csv(out_path, index=False)
print(f"\nEDA sample saved → {out_path}")